# 1일차 파트 1 실습 풀이

`1일차_실습.ipynb`의 풀이와 해설입니다. 판정과 이유는 하나의 예시입니다. 모델의 응답은 실행할 때마다 다르므로 여러분의 판정이 풀이와 다를 수 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 받지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. Windows PowerShell에서는 다음과 같이 설정한 뒤 터미널을 새로 엽니다.

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. 이 값이 없으면 노트북은 Anthropic API를 쓰는 것으로 봅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 마지막 줄에 `준비 완료`가 나오면 연결된 것입니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/1일차'


def read_material(name):
    """재료 폴더(data/1일차)의 파일을 읽어 문자열로 돌려줍니다."""
    return open(f'{DATA}/{name}', encoding='utf-8').read().strip()


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ... 가 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `401 auth` | 키가 틀렸습니다. 앞뒤 공백이나 따옴표 없이 다시 넣습니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `NameError` | 앞 셀을 건너뛰었습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 1 질문 카드로 사용 기준표 만들기

**목적** 질문 유형마다 모델의 응답을 어떻게 쓸지 직접 정해 봅니다. 응답이 맞았는지와 응답을 어떻게 쓸지는 별개라는 점을 확인합니다.

**재료** `data/1일차/질문카드.txt`. 정의, 원리, 계산, 최신 정보, 사내 규격 순서의 질문 다섯 개입니다. 교재의 예제 1-6과 유형은 같고 주제가 다릅니다.

**할 일**

1. 계산 카드의 정답을 코드로 구합니다. 모델에 보내기 전에 정답을 먼저 알아야 응답을 판정할 수 있습니다.
2. 카드 다섯 장을 소형 모델과 기본 모델에 보냅니다(준비된 셀을 실행합니다).
3. 응답을 읽고 사용 기준표 `table`을 채웁니다. 모델별 판정, 사용 기준, 이유 한 줄입니다.
4. 최신 정보 카드에 "근거를 확인할 수 없는 내용은 '확인 불가'라고 적어줘" 한 줄을 붙여 다시 보내고, 응답이 어떻게 달라졌는지 적습니다.
5. 사내 규격 카드의 응답에서 그대로 쓰면 위험한 문장을 하나 골라 적습니다.

**완료 기준** 테스트 셀이 "실습 1 통과"를 출력합니다. 4번과 5번은 "적어 보기" 셀에 적습니다.

<details><summary>힌트 1</summary>

누적 수율은 단계별 수율을 모두 곱한 값입니다. 퍼센트를 100으로 나누어 곱한 뒤 다시 100을 곱하고, `round(값, 2)`로 소수 둘째 자리까지 구합니다.
</details>

<details><summary>힌트 2</summary>

판정은 응답이 맞았는지입니다. 최신 정보처럼 맞는지 확인할 근거가 응답 밖에 있으면 "판정 불가"입니다. 사용 기준은 응답이 맞았는지가 아니라 질문의 유형으로 정합니다. 교재의 "질문 유형별 사용 기준" 절의 표를 참고합니다.
</details>

In [3]:
# 실습 1 재료 (그대로 실행합니다)
import checks_d1

cards = {}
for block in read_material('질문카드.txt').split('\n\n'):
    title, _, body = block.partition('\n')
    cards[title.strip('[]')] = body
for kind, card in cards.items():
    print(f'[{kind}] {card}\n')

[정의] 포토 공정에서 오버레이(overlay) 오차가 무엇인지, 왜 생기는지 세 문장으로 설명해줘.

[원리] CVD 증착에서 온도를 올리면 증착 속도가 처음에는 빠르게 늘다가 어느 온도 이상에서는 거의 늘지 않는 이유를 세 문장으로 설명해줘.

[계산] 공정 네 단계의 단계별 수율이 98.5%, 97.2%, 99.1%, 96.8% 일 때 누적 수율을 계산해줘.
답 형식: 첫 줄에 결과(%, 소수 둘째 자리까지)만, 그다음 계산 과정을 3줄 이내로.

[최신 정보] 2026년 11월 현재 출하 중인 최신 EUV 노광 장비의 모델명과 해상도를 알려줘.

[사내 규격] 우리 라인 노광 장비의 오버레이 관리 한계값과 한계값을 넘었을 때의 조치 절차를 알려줘.



In [4]:
# 실습 1 (1) 계산 카드의 정답을 코드로 구합니다
def cumulative_yield(step_yields):
    """단계별 수율(%)의 목록을 받아 누적 수율(%)을 소수 둘째 자리까지 돌려줍니다."""
    total = 1.0
    for step_yield in step_yields:
        total *= step_yield / 100
    return round(total * 100, 2)


correct_yield = cumulative_yield([98.5, 97.2, 99.1, 96.8])
print('코드로 구한 정답:', correct_yield, '%')

코드로 구한 정답: 91.84 %


In [5]:
# 실습 1 (2) 카드 다섯 장을 두 모델에 보냅니다 (실제 API 를 부릅니다. 10회)
for kind, card in cards.items():
    for model in ['소형', '기본']:
        print(f'===== {kind} · {model} 모델 =====')
        llm.Chat(model=model).say(card)
        print()

===== 정의 · 소형 모델 =====


오버레이(overlay) 오차는 반도체 포토 공정에서 이전 층에 형성된 패턴과 현재 노광하는 층의 패턴이 설계된 위치에 정확히 겹치지 않고 어긋나는 정도를 말합니다. 이 오차는 웨이퍼 스테이지의 위치 제어 정밀도 한계, 웨이퍼의 열팽창이나 변형(휨, 응력), 이전 공정(식각, 증착, CMP 등)에서 생긴 기판 왜곡 등 여러 요인이 복합적으로 작용하여 발생합니다. 오버레이가 허용 범위를 벗어나면 콘택 불량, 단락이나 개방 같은 전기적 결함과 수율 저하로 이어지므로, 공정에서는 정렬 마크 측정과 보정(alignment/correction)으로 이를 지속적으로 관리합니다.

===== 정의 · 기본 모델 =====


오버레이 오차는 반도체 포토 공정에서 이전 공정으로 형성된 패턴 위에 새로 노광한 패턴이 설계된 위치와 어긋나는 정도를 말하며, 보통 nm 단위로 측정합니다. 이는 노광 장비의 스테이지 위치 제어 오차, 렌즈 수차에 의한 왜곡, 정렬 마크 측정 오차 등 장비적 요인과, 열처리나 박막 응력으로 인한 웨이퍼 변형, 마크 비대칭 같은 공정적 요인이 겹쳐 발생합니다. 오버레이 오차가 커지면 하부 층과 상부 층의 배선이나 콘택이 제대로 연결되지 않아 단선, 단락 등 불량이 생기므로, 미세 공정일수록 이를 수 nm 이하로 관리하는 것이 수율 확보에 핵심입니다.

===== 원리 · 소형 모델 =====


저온에서는 표면 화학반응이 속도를 결정하므로 온도가 오를수록 반응속도가 아레니우스식(exp(-Ea/kT))으로 급격히 증가한다. 고온에서는 반응이 너무 빨라져 기체 전구체가 기판 표면의 경계층을 통해 확산으로 공급되는 속도가 율속 단계가 되는데, 확산계수는 온도에 약하게만 의존하므로 증착 속도가 거의 늘지 않고 포화된다. 따라서 CVD 공정은 온도에 따라 표면반응 제한 영역에서 물질전달 제한 영역으로 전환되며, 이 전환점 이상에서는 온도보다 가스 유량이나 압력 같은 물질전달 조건이 증착 속도를 좌우하게 된다.

===== 원리 · 기본 모델 =====


저온 영역에서는 표면에서 일어나는 화학 반응이 속도를 좌우하는 반응 律速(surface-reaction-limited) 상태이므로, 증착 속도가 아레니우스 식(k ∝ exp(−Ea/RT))을 따라 온도에 지수적으로 빠르게 증가합니다. 온도가 더 높아지면 표면 반응이 충분히 빨라져서, 반응물이 기체 상태에서 기판 표면까지 확산해 오는 속도가 전체 속도를 결정하는 물질 전달 律速(mass-transport-limited) 상태로 바뀝니다. 이 영역에서는 반응물 공급량이 한계이고 확산 계수의 온도 의존성도 약하기 때문에 증착 속도가 온도에 거의 무관하게 포화되며, 더 높은 온도에서는 기상 반응이나 반응물 고갈 때문에 오히려 감소하기도 합니다.

(참고: 위 문장에서 '律速'은 '율속', 즉 속도를 결정하는 단계라는 뜻입니다.)

===== 계산 · 소형 모델 =====


**91.84%**

0.985 × 0.972 × 0.991 × 0.968 = 0.91844… 이므로 누적 수율은 약 91.84%입니다.
(앞의 두 단계 0.95742 → 세 번째 단계 0.94880 → 네 번째 단계 0.91844)

===== 계산 · 기본 모델 =====


91.84%

1. 0.985 × 0.972 = 0.95742, 여기에 × 0.991 = 0.948803
2. 0.948803 × 0.968 ≈ 0.918442
3. 누적 수율 = 0.918442 × 100 ≈ 91.84%

===== 최신 정보 · 소형 모델 =====


2026년 11월 시점의 출하 현황은 제가 확실하게 알 수 없습니다. 제 학습 데이터는 그 이전에 끝나며, 출하 일정은 자주 바뀌므로 최신 정보는 ASML 공식 발표나 실적 발표를 확인하시는 것이 정확합니다. 제가 알고 있는 범위에서 정리하면 다음과 같습니다.

**일반(Low-NA) EUV 장비**
- **NXE:3600D**: 개구수(NA) 0.33, 해상도 약 13nm(하프피치 기준)
- **NXE:3800E**: NA 0.33, 시간당 웨이퍼 처리량(throughput)을 높인 후속 모델

**High-NA EUV 장비**
- **EXE:5000**: NA 0.55, 해상도 약 8nm 수준을 목표로 한 개발·양산 검증용 모델
- **EXE:5200 (B 등 파생 모델 포함)**: 상용 양산을 목표로 한 모델로, 2025~2026년경부터 인텔 등 고객사에 출하가 본격화될 것으로 알려져 있습니다.

해상도는 NA 값에 비례하므로 High-NA(0.55)가 Low-NA(0.33)보다 미세한 패턴을 구현합니다. 정확한 현재 출하 모델과 사양은 ASML 홈페이지의 제품 페이지나 분기별 실적 자료에서 확인하시기 바랍니다.

===== 최신 정보 · 기본 모델 =====


**2026년 11월 시점의 출하 현황은 제가 확인할 수 없습니다.** 제가 아는 정보는 그 이전까지이고, 아래는 ASML(EUV 노광 장비 유일 공급사) 기준입니다.

## 1. 양산 주력: 0.33 NA EUV
- **모델: TWINSCAN NXE:3800E**
- 해상도: 약 **13 nm** (dense line, 피치 26 nm급)
- 생산성: 시간당 약 **195~220장** (wph)
- 오버레이(Matched Machine): 약 1.1 nm 수준
- 용도: 3 nm/2 nm급 로직, 최신 DRAM의 EUV 레이어 양산

## 2. 차세대: High-NA EUV (0.55 NA)
- **모델: TWINSCAN EXE:5000** (R&D용 초기 모델), **EXE:5200B** (양산용 모델)
- 해상도: 약 **8 nm**
- 생산성: EXE:5200B는 약 **175 wph** 목표
- 오버레이: 약 0.7 nm 목표
- 인텔이 가장 먼저 도입했고, TSMC, 삼성, SK하이닉스 등은 R&D나 도입 초기 단계로 알려져 있습니다. 양산 적용 시점은 업체마다 다릅니다.

## 참고
- 2026년 11월 기준으로 새 모델(예: NXE 후속 모델)이 발표되었거나 EXE:5200B의 출하 규모가 달라졌을 수 있습니다.
- 정확한 최신 정보는 ASML 공식 사이트의 제품 페이지나 최근 분기 실적 발표 자료에서 확인하시길 권합니다.

특정 항목(예: 업체별 도입 현황, High-NA 양산 적용 시점)을 더 자세히 알려드릴 수도 있습니다.

===== 사내 규격 · 소형 모델 =====


# 노광 장비 오버레이 관리 한계값 및 이상 시 조치 절차

사내 SOP와 제품 스펙이 최종 기준이므로 아래 값은 **일반적인 구조와 업계 통상 관리 방식**입니다. 실제 수치는 공정 노드, 레이어, 제품별로 다르니 사내 문서로 확인하고 대입하세요.

---

## 1. 관리 지표 정의

| 항목 | 내용 |
|---|---|
| **Overlay (X, Y)** | 현 레이어 패턴과 하부 레이어 패턴의 위치 오차 (nm) |
| **Mean (μ)** | 웨이퍼/샷 평균 오차 (시스템 오프셋 판단) |
| **3σ** | 산포 지표, 보통 관리 지표로 사용 |
| **Mean+3σ 또는 |μ|+3σ** | 대부분의 fab에서 실제 판정에 쓰는 값 |
| **Residual (Mag, Rot, Ortho 등)** | 배율, 회전, 직교도 등 고차 성분 |

---

## 2. 한계값 체계 (3단계 권장 구조)

| 단계 | 기준 예시 | 의미 |
|---|---|---|
| **① 목표(Target)** | 공정 능력 기준 설정값 | 정상 운영 목표 |
| **② 경고(Warning / UCL)** | 예: Spec의 60~70% 수준 (3σ 관리한계) | 추세 감시, 모니터링 강화 |
| **③ 조치(Action / Spec Limit)** | 제품 스펙 허용치 (USL) | 즉시 Lot Hold 및 조치 |

- **공정능력 지수 Cpk ≥ 1.33** (일반), 핵심 레이어는 **≥ 1.67** 목표
- 관리도: **X-bar / R 또는 I-MR 차트**로 Warning/Action 판정 병행
- 예시 판정식: `|Mean| + 3σ > Spec 50%` → Warning, `|Mean| + 3σ > Spec` → Action

> 참고로 최신 미세 노드의 오버레이 스펙은 수 nm 수준이고, 성숙 노드는 수십 nm 수준입니다. 실제 값은 반드시 사내 스펙으로 확인하세요.

---

## 3. 한계 초과 시 조치 절차

### Step 1. 즉시 대응 (

# 오버레이 관리 한계값과 이탈 시 조치 절차

우리 라인의 실제 관리 한계값과 승인된 OCAP(Out of Control Action Plan)은 제가 확인할 수 없습니다. 공정, 레이어, 장비(스캐너/스테퍼), 디바이스 세대마다 다르고 회사 내부 문서로 관리되기 때문입니다. 추측한 숫자를 드리면 오히려 위험하므로, 확인할 위치와 일반적인 관리 체계, 조치 절차 템플릿을 정리해 드립니다.

---

## 1. 실제 한계값 확인처리

| 확인 대상 | 문서/시스템 |
|---|---|
| 레이어별 오버레이 스펙 | 공정 스펙서, Design Rule, 레이어별 Overlay Budget |
| 장비별 관리 한계 | 장비 관리 기준서, SPC 시스템 차트 설정값 |
| 이탈 시 조치 절차 | OCAP 문서, 노광 공정 표준(SOP) |
| 보정(APC/R2R) 한계 | APC 모델 설정, 보정량 상·하한 |

담당 부서는 보통 포토 공정기술, 포토 장비기술, 계측(Metrology) 파트입니다.

---

## 2. 일반적인 한계값 체계

**스펙 한계(USL/LSL)와 관리 한계(UCL/LCL)를 구분**하는 것이 기본입니다.

- **오버레이 스펙**: 보통 해당 레이어 최소 CD(또는 디자인 룰)의 약 15~25% 수준에서 설정합니다. 예를 들어 CD가 수십 nm인 레이어는 수 nm 수준입니다.
- **관리 지표**: X/Y 방향 각각의 `|Mean| + 3σ`, 잔차(Residual), 보정 가능 성분(Translation, Rotation, Magnification, Orthogonality 등)
- **관리 한계**: 스펙보다 안쪽에 설정합니다. 예: 스펙의 70~80% 수준을 Action Limit, 그보다 안쪽을 Warning Limit으로 둡니다.
- **SPC 룰**: 한계 초과뿐 아니라 연속 상승/하강, 한쪽 치우침(Western Electric/Nelson 룰) 등 추세 이상도 감시합니다.

---

## 3. 한계 이탈 시 일반적인 조치 

In [6]:
# 실습 1 (3) 응답을 읽고 사용 기준표를 채웁니다
# 판정: '맞음' · '틀림' · '판정 불가'
# 기준: '그대로 쓴다' · '검증하고 쓴다' · '도구에 맡긴다' · '입력하지 않는다'
table = {
    '정의':      {'소형': '맞음', '기본': '맞음', '기준': '그대로 쓴다',
                  '이유': '겹쳐 찍는 층 사이의 정렬 오차라는 정의는 읽고 판정할 수 있다'},
    '원리':      {'소형': '맞음', '기본': '맞음', '기준': '그대로 쓴다',
                  '이유': '저온에서는 반응 속도가, 고온에서는 원료 공급이 속도를 정한다는 일반 원리다'},
    '계산':      {'소형': '맞음', '기본': '맞음', '기준': '도구에 맡긴다',
                  '이유': '코드로 구한 91.84% 와 대조해야 맞았는지 알 수 있다'},
    '최신 정보': {'소형': '판정 불가', '기본': '판정 불가', '기준': '검증하고 쓴다',
                  '이유': '모델명과 해상도는 제조사 자료와 대조하기 전에는 맞는지 알 수 없다'},
    '사내 규격': {'소형': '판정 불가', '기본': '판정 불가', '기준': '입력하지 않는다',
                  '이유': '우리 라인의 한계값은 모델이 알 수 없고, 덧붙인 일반론은 우리 기준이 아니다'},
}

In [7]:
# 실습 1 확인 (API 를 부르지 않습니다)
checks_d1.check_table(table, correct_yield)

1. 통과 계산 카드의 정답이 91.84% 다
2. 통과 기준표에 질문 유형 다섯 가지가 모두 있다
3. 통과 모델별 판정이 '맞음', '틀림', '판정 불가' 가운데 하나다
4. 통과 사용 기준이 네 가지 가운데 하나다
5. 통과 유형마다 이유를 적었다
6. 통과 계산은 모델이 맞혔더라도 도구에 맡긴다
7. 통과 사내 규격은 입력하지 않는다
실습 1 통과


[(True, '계산 카드의 정답이 91.84% 다'),
 (True, '기준표에 질문 유형 다섯 가지가 모두 있다'),
 (True, "모델별 판정이 '맞음', '틀림', '판정 불가' 가운데 하나다"),
 (True, '사용 기준이 네 가지 가운데 하나다'),
 (True, '유형마다 이유를 적었다'),
 (True, '계산은 모델이 맞혔더라도 도구에 맡긴다'),
 (True, '사내 규격은 입력하지 않는다')]

In [8]:
# 실습 1 (4) 최신 정보 카드에 한 줄을 붙여 다시 보냅니다 (실제 API 를 부릅니다. 2회)
rule = "\n근거를 확인할 수 없는 내용은 '확인 불가'라고 적어줘."
for model in ['소형', '기본']:
    print(f'===== 최신 정보 + 한 줄 · {model} 모델 =====')
    llm.Chat(model=model).say(cards['최신 정보'] + rule)
    print()

===== 최신 정보 + 한 줄 · 소형 모델 =====


## 결론

**2026년 11월 현재 출하 중인 최신 EUV 장비의 모델명과 해상도는 제가 확인할 수 없습니다.** 제 정보는 특정 시점까지만 반영되어 있고, 그 이후의 출하 현황이나 신규 모델 발표는 확인 불가입니다.

## 제가 알고 있는 범위 (2026년 11월 이전 기준)

| 모델 | 개구수(NA) | 해상도(ASML 공개 사양 기준) | 출하 상태 |
|---|---|---|---|
| NXE:3400C | 0.33 | 13nm | 양산 라인 설치 이력 있음 (확인됨) |
| NXE:3600D | 0.33 | 13nm | 확인 불가 (출하 시점·수량 근거 불충분) |
| NXE:3800E | 0.33 | 13nm (생산성 개선형으로 발표됨) | 출하 시점 확인 불가 |
| EXE:5000 | 0.55 (High-NA) | 8nm | 개발·설치 단계 이력 있음, 양산 출하 여부 확인 불가 |
| EXE:5200 | 0.55 (High-NA) | 8nm (발표 기준) | 출하 시점 확인 불가 |

## 참고 사항

- **해상도 수치**는 ASML이 공개한 사양 기준이며, 실제 공정 노드(예: 2nm급 등)와 1:1로 대응하지 않습니다. 실제 패턴 해상도는 마스크, 레지스트, 공정 조건에 따라 달라집니다.
- **0.55 NA(High-NA)** 장비의 양산 출하 시점과 고객사별 설치 현황은 제가 확인할 수 없습니다.
- 모델명 체계(예: 3400, 3600, 3800, 5000, 5200 등)는 제가 아는 범위의 정보이며, 2026년 이후 신규 모델은 확인 불가입니다.

## 정확한 최신 정보 확인 방법

1. **ASML 공식 사이트 및 보도자료**: 제품 라인업과 사양이 가장 직접적인 1차 출처입니다.
2. **ASML 분기 실적 발표 자료**: EUV 출하 대수와 제품 믹스가 공개됩니다.
3. **SEMI, IMEC, Intel·TSMC·Samsung·SK하이닉스 기술 발표**: 고객사 도입 현황 확인에 유용합니다.

원하시면 특정 모델(예: 

## 결론
2026년 11월 시점의 "현재 출하 중인 최신 모델"은 **확인 불가**입니다. 제 지식은 그 시점까지 갱신되어 있지 않아서, 제가 아는 마지막 공개 정보까지만 말씀드립니다.

## 제가 아는 ASML EUV 장비 (공개 자료 기준)

| 구분 | 모델명 | NA | 해상도 | 비고 |
|---|---|---|---|---|
| Low-NA 양산 | **NXE:3800E** | 0.33 | 약 13 nm | 처리량 약 195~220 wph(웨이퍼/시간) 수준으로 발표 |
| High-NA 초기/연구 | **EXE:5000** | 0.55 | 약 8 nm | 초기 평가용, 주로 R&D |
| High-NA 양산 지향 | **EXE:5200B** | 0.55 | 약 8 nm | 양산용으로 소개, 처리량 약 175 wph 안팎 |

## 확인 불가 항목
- **2026년 11월 기준 실제 출하 모델**: 확인 불가. 이후 개량형(예: NXE:3800E의 후속)이나 새 High-NA 모델이 나왔을 수 있습니다.
- **EXE:5200B의 고객별 출하·양산 투입 현황**: 확인 불가.
- **각 모델의 최신 처리량·오버레이 수치**: 확인 불가. 위 수치는 발표 당시 목표나 사양일 수 있습니다.
- **Hyper-NA(0.75 NA급) 장비**: 개발 검토 단계로 알고 있으며, 출하 여부는 확인 불가입니다.

## 참고
"해상도"는 NA와 공정 계수(k1)에 따라 달라지는 이론상 값입니다. 실제 양산 패턴 피치는 공정 조건에 따라 다르므로, 정확한 최신 사양은 ASML 공식 제품 페이지나 최근 분기 실적 발표 자료로 확인하시기 바랍니다.



**해설**

아래는 2026-11-19에 Claude 모델로 실행한 결과입니다. 여러분의 응답은 다를 수 있습니다.

(풀이를 실행한 뒤 채웁니다)

- **계산 정답을 먼저 구합니다.** 정답을 모르고 응답을 받으면 91.84%와 91.87%를 구분할 수 없습니다. 모델이 맞혔더라도 맞혔다는 것을 안 것은 코드로 계산한 값이 있었기 때문입니다. 그래서 계산의 사용 기준은 응답과 관계없이 "도구에 맡긴다"입니다.
- **최신 정보는 판정할 근거가 응답 밖에 있습니다.** 모델명과 수치가 그럴듯해도 제조사의 공식 자료와 대조하기 전에는 맞는지 알 수 없습니다. "확인 불가" 규칙은 모델이 추정으로 채우는 것을 줄여 주지만, 남은 내용이 맞다는 뜻은 아닙니다.
- **사내 규격의 응답에서 위험한 것은 일반론입니다.** 모델은 우리 라인의 값을 모른다고 밝힌 뒤 "일반적으로는" 하고 덧붙입니다. 이 부분만 옮기면 우리 라인의 기준처럼 읽힙니다. 사내 규격은 사내 문서에서 찾습니다.